# 8번 유압 밸브 상태 4단계 분류
- 후보 A: 9.0초 이후 압력이 처음 20 bar를 넘는 샘플 번호 → 클래스 범위 중간점 경계
- 후보 B: 9.0~10.0초 원시 압력 100개 → RandomForest (교차 확인)
- 채택 규칙: 모든 분할에서 동점이면 A. test 진단은 정보 제공용이며 제출을 바꾸지 않음

In [1]:
import time; t0 = time.time()
import numpy as np, pandas as pd
from sklearn.metrics import f1_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

d = np.load('data/sensor_data.npz'); ids = d['cycle_id']
tr = pd.read_csv('data/train.csv', encoding='utf-8-sig')
te = pd.read_csv('data/test.csv', encoding='utf-8-sig')
idx = {c: i for i, c in enumerate(ids)}
ti = tr.cycle_id.map(idx).values; si = te.cycle_id.map(idx).values; y = tr.label.values
P, F, T, V = d['pressure'], d['flow'], d['temperature'], d['vibration']
assert not np.isnan(ti.astype(float)).any() and not np.isnan(si.astype(float)).any()
print(tr.shape, te.shape, np.bincount(y))

/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


(1764, 3) (441, 1) [904 290 290 280]


## 후보 A: 압력 재상승 시점

In [2]:
def rise(thr=20):
    # 9.0s(=900) 이후 처음 thr bar를 넘는 샘플 번호
    seg = P[:, 900:1200]
    assert (seg > thr).any(1).all(), 'threshold never crossed'
    return np.argmax(seg > thr, 1) + 900

def fit_bounds(r, mask):
    # 클래스 범위 중간점
    rg = [(r[ti][mask & (y == k)].min(), r[ti][mask & (y == k)].max()) for k in range(4)]
    return [(rg[k][1] + rg[k + 1][0]) / 2 for k in range(3)]

r = rise(20)
bounds = fit_bounds(r, np.ones(len(ti), bool))
predA = np.digitize(r, bounds)
for k in range(4):
    print(k, 'rise range', r[ti][y == k].min(), r[ti][y == k].max())
print('bounds', bounds, '| train macro-F1', f1_score(y, predA[ti], average='macro'))

0 rise range 936 938
1 rise range 942 945
2 rise range 952 955
3 rise range 964 968
bounds [940.0, 948.5, 959.5] | train macro-F1 1.0


## 안전본 저장 (A)

In [3]:
def save(pred):
    sub = pd.DataFrame({'cycle_id': te.cycle_id, 'label': pred[si]})
    sub.to_csv('submission.csv', index=False, encoding='utf-8-sig')
    return sub
sub = save(predA)
print(sub.label.value_counts().sort_index().to_dict(), f'{time.time() - t0:.1f}s')

{0: 221, 1: 70, 2: 70, 3: 80} 0.8s


## 검증 분할: fit→valid, 온도 분할, 운전조건 군집 LOGO

In [4]:
fit = tr.split.values == 'fit'
tm = T.mean(1)
X_cond = np.c_[tm, T[:, -1] - T[:, 0], V.mean(1), P[:, 1100:1800].mean(1), P[:, 5000:].mean(1), F[:, 110:180].mean(1)]
g = KMeans(6, n_init=10, random_state=0).fit_predict(StandardScaler().fit_transform(X_cond))
print('cluster sizes train', np.bincount(g[ti], minlength=6), 'test', np.bincount(g[si], minlength=6))

splits = [('fit->valid', fit, ~fit),
          ('cool->hot', tm[ti] < 45, tm[ti] >= 50),
          ('hot->cool', tm[ti] >= 50, tm[ti] < 45)]
splits += [(f'LOGO g{G}', g[ti] != G, g[ti] == G) for G in range(6) if (g[ti] == G).sum() >= 20]

cluster sizes train [632 280 163 661  10  18] test [115  34 208  81   3   0]


In [5]:
W = P[:, 900:1000]
rows = []
for name, trn, tst in splits:
    bA = fit_bounds(r, trn)
    pA = np.digitize(r[ti][tst], bA)
    margin = np.min(np.abs(r[ti][tst][:, None] - np.array(bA)[None]), 1).min()
    B = RandomForestClassifier(300, n_jobs=-1, random_state=0).fit(W[ti][trn], y[trn])
    pB = B.predict(W[ti][tst])
    rows.append([name, tst.sum(), f1_score(y[tst], pA, average='macro'), f1_score(y[tst], pB, average='macro'), margin])
res = pd.DataFrame(rows, columns=['split', 'n', 'F1_A', 'F1_B', 'min_margin_A'])
res['diff_B-A'] = res.F1_B - res.F1_A
print(res.round(4).to_string(index=False))
print('B-A mean', res['diff_B-A'].mean(), '| B wins', (res['diff_B-A'] > 0).sum(), '/ A wins', (res['diff_B-A'] < 0).sum())

     split   n  F1_A   F1_B  min_margin_A  diff_B-A
fit->valid 440   1.0 1.0000           2.0    0.0000
 cool->hot 464   1.0 0.9432           1.0   -0.0568
 hot->cool 974   1.0 1.0000           1.5    0.0000
   LOGO g0 632   1.0 1.0000           3.0    0.0000
   LOGO g1 280   1.0 1.0000           2.5    0.0000
   LOGO g2 163   1.0 1.0000           1.5    0.0000
   LOGO g3 661   1.0 1.0000           1.5    0.0000
B-A mean -0.00811688311688311 | B wins 0 / A wins 1


## test 진단 (정보 제공용, 제출에 영향 없음)

In [6]:
for thr in [5, 10, 40, 80]:
    rr = rise(thr)
    pp = np.digitize(rr, fit_bounds(rr, np.ones(len(ti), bool)))
    print(f'thr {thr:>2} bar: test agreement with A {np.mean(pp[si] == predA[si]):.3f}')
B_full = RandomForestClassifier(300, n_jobs=-1, random_state=0).fit(W[ti], y)
predB = B_full.predict(W)
dis = te.cycle_id[predB[si] != predA[si]].tolist()
print('A vs B test agreement', np.mean(predB[si] == predA[si]), '| disagreements', dis)
m = np.min(np.abs(r[si][:, None] - np.array(bounds)[None]), 1)
print('test margin counts', dict(zip(*np.unique(m, return_counts=True))), '| margin<2:', (m < 2).sum())

thr  5 bar: test agreement with A 1.000
thr 10 bar: test agreement with A 1.000
thr 40 bar: test agreement with A 1.000
thr 80 bar: test agreement with A 1.000


A vs B test agreement 1.0 | disagreements []
test margin counts {2.0: 3, 3.0: 81, 3.5: 10, 4.0: 207, 4.5: 94, 5.5: 43, 6.5: 1, 7.5: 1, 8.5: 1} | margin<2: 0


## 채택 판정 → 최종 제출

In [7]:
use_B = res['diff_B-A'].mean() > 0 and (res['diff_B-A'] > 0).sum() > len(res) / 2
final = predB if use_B else predA
print('adopted:', 'B' if use_B else 'A')
sub = save(final)
chk = pd.read_csv('submission.csv', encoding='utf-8-sig')
assert len(chk) == 441 and (chk.cycle_id == te.cycle_id).all() and set(chk.label) <= {0, 1, 2, 3}
assert list(chk.columns) == ['cycle_id', 'label']
print(chk.label.value_counts().sort_index().to_dict(), f'total {time.time() - t0:.1f}s')

adopted: A
{0: 221, 1: 70, 2: 70, 3: 80} total 5.3s
